<a href="https://colab.research.google.com/github/WaitUps/CourseraExercises-AI-Agents-and-Agentic-AI-Course/blob/main/Module%201/Part2_Building_a_Simple_AI_Agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Part 2: Building a Simple AI Agent - The Agent Loop (OpenAI to Gemini Adaptation)

Now that you understand the agent loop and how to craft effective prompts, we can build a simple AI agent. This agent will be able to list files in a directory, read their content, and answer questions about them. We’ll break down the agent loop—how it receives input, decides on actions, executes them, and updates its memory—step by step.

***
---

### Gemini API Configuration and Libraries

This section sets up the necessary libraries and configures the Google Gemini API. It imports `userdata` to securely access your API key stored in Colab secrets, and `google.genai` for interacting with the Gemini models. The `json` library is also imported for structured data handling within the agent loop.

In [1]:
# Migrated from OpenAI to Gemini model.
# Import necessary libraries
from google.colab import userdata # Imports the userdata module to access secrets stored in Google Colab
import google.genai as genai # Imports the Google Generative AI SDK
import json # Required for json.dumps in the agent loop and parsing actions


# Configure Gemini API
# To use the Gemini API, you'll need an API key. If you don't already have one, create a key in Google AI Studio.
# In Colab, add the key to the secrets manager under the "🔑" in the left panel. Give it the name `GOOGLE_API_KEY`.
GOOGLE_API_KEY = userdata.get('GOOGLE_API_KEY')
client = genai.Client(api_key=GOOGLE_API_KEY) # Initialize the client with the API key

This code block lists the available Gemini models that support the `generateContent` method. This is important for understanding which models can be used for text generation and conversational AI tasks within the configured API environment.

In [2]:
#import google.genai as genai

for m in client.models.list():
  # Check if the model supports the 'generateContent' method.
  # This is crucial for models that are intended for text generation and conversational AI.
  if "generateContent" in m.supported_actions:
    print(m.name)

models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.0-flash
models/gemini-2.0-flash-001
models/gemini-2.0-flash-lite-001
models/gemini-2.0-flash-lite
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-3-1b-it
models/gemma-3-4b-it
models/gemma-3-12b-it
models/gemma-3-27b-it
models/gemma-3n-e4b-it
models/gemma-3n-e2b-it
models/gemma-4-26b-a4b-it
models/gemma-4-31b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3-pro-preview
models/gemini-3-flash-preview
models/gemini-3.1-pro-preview
models/gemini-3.1-pro-preview-customtools
models/gemini-3.1-flash-lite-preview
models/gemini-3-pro-image-preview
models/nano-banana-pro-preview
models/gemini-3.1-flash-image-preview
models/lyria-3-clip-preview
models/lyria-3-pro-preview
models/gemini-robotics-er-1.5-preview
models/gemini-2.5-computer-use-preview-10-2025
models/deep-research-pro-preview-12-2

***
---


### `generate_response` Function

The `generate_response` function is crucial for interfacing with the Gemini LLM. It takes a list of messages, formats them appropriately for the `genai.Client` (including converting 'system' roles to 'user' for context setting), and then sends them to the specified Gemini model to generate a response. This function also provides a detailed log of the API call for debugging and transparency.

In [3]:
# Migrated from OpenAI to Gemini model.
from typing import List, Dict # Used for type hinting to improve code readability and maintainability
import google.genai as genai # Imports the Google Generative AI SDK
import json # Import json for pretty printing

# The 'client' object is expected to be globally available from the API key setup cell.
# It provides the direct interface for interacting with the Gemini API service.

def generate_response(messages: List[Dict]) -> tuple[str, str]:
    """
    Calls the Gemini LLM to generate a response based on a list of messages using the direct client interface.
    It handles system instructions by incorporating them as the first 'user' message in the content list.
    Returns a tuple: (generated_text: str, detailed_markdown_log: str).
    """

    model_to_use = 'gemini-flash-lite-latest' #Other options: 'gemini-flash-latest' or 'gemma-3-1b-it'
    generation_config_dict = {'max_output_tokens':3072}


    detailed_markdown_log_parts = []
    detailed_markdown_log_parts.append("#### `generate_response` Call Details\n\n")

    detailed_markdown_log_parts.append("**Input Messages:**\n")
    detailed_markdown_log_parts.append(f"```json\n{json.dumps(messages, indent=2)}\n```\n\n")

    formatted_messages_for_client = []
    for msg in messages:
        # CRITICAL ADAPTATION FOR `client.models.generate_content`:
        # This specific method of the Gemini API does not have a distinct 'system_instruction' parameter.
        # Instead, system-level instructions or persona settings must be provided as the first 'user' turn.
        # Therefore, if the input message has a 'system' role, we convert it to a 'user' role
        # for compatibility with the API call. This effectively sets the context for the model.
        actual_role = msg['role']
        if actual_role == 'system':
            # Convert 'system' role to 'user' role for API compatibility.
            # This ensures that system instructions are correctly interpreted by the model
            # as part of the conversation history, setting the overall context or persona.
            actual_role = 'user'
        formatted_messages_for_client.append({'role': actual_role, 'parts': [{'text': msg['content']}]})

    detailed_markdown_log_parts.append("**Formatted Messages for `client.models.generate_content`:**\n")
    detailed_markdown_log_parts.append(f"```json\n{json.dumps(formatted_messages_for_client, indent=2)}\n```\n\n")


    detailed_markdown_log_parts.append(f"**Model Used:** `{model_to_use}`\n\n")
    detailed_markdown_log_parts.append("**Generation Configuration:**\n")
    detailed_markdown_log_parts.append(f"```json\n{json.dumps(generation_config_dict, indent=2)}\n```\n\n")

    # Content Generation using the direct client interface:
    # Instead of instantiating a GenerativeModel, we call `generate_content` directly on `client.models`.
    # The 'model' argument specifies which Gemini model to use.
    # 'contents' is the list of formatted messages forming the conversation history.
    # `generation_config` controls generation parameters like `max_output_tokens`.
    response = client.models.generate_content(
        model=model_to_use, # Specify the model directly
        contents=formatted_messages_for_client,
        config=generation_config_dict
    )

    detailed_markdown_log_parts.append("**LLM Raw Response (text content):**\n")
    detailed_markdown_log_parts.append(f"\n{response.text}\n\n\n")

    final_markdown_output = "".join(detailed_markdown_log_parts)

    # Return Value:
    # The generated text content from the model's response is extracted and returned
    # along with the detailed markdown log.
    return response.text, final_markdown_output

### The Agent Loop in Python

The agent loop is the backbone of our AI agent, enabling it to perform tasks by combining response generation, action execution, and memory updates in an iterative process. This section focuses on how the agent loop works and its role in making the agent dynamic and adaptive.

*   **Construct Prompt**: Combine the agent’s memory, user input, and system rules into a single prompt. This ensures the LLM has all the context it needs to decide on the next action, maintaining continuity across iterations.
*   **Generate Response**: Send the constructed prompt to the LLM and retrieve a response. This response will guide the agent’s next step by providing instructions in a structured format.
*   **Parse Response**: Extract the intended action and its parameters from the LLM’s output. The response must adhere to a predefined structure (e.g., JSON format) to ensure it can be interpreted correctly.
*   **Execute Action**: Use the extracted action and its parameters to perform the requested task with the appropriate tool. This could involve listing files, reading content, or printing a message.
*   **Convert Result to String**: Format the result of the executed action into a string. This allows the agent to store the result in its memory and provide clear feedback to the user or itself.
*   **Continue Loop?**: Evaluate whether the loop should continue based on the current action and results. The loop may terminate if a “terminate” action is specified or if the agent has completed the task.

The agent iterates through this loop, refining its behavior and adapting its actions until it reaches a stopping condition. This process is what enables the agent to interact dynamically and respond intelligently to tasks.

Here’s how these steps come together in code:

In [4]:
# Initialize variables for the agent loop
iterations = 0
max_iterations = 5 # Set a reasonable limit to prevent infinite loops
memory = [] # Stores the conversation history for context

In [5]:
# Migrated from OpenAI to Gemini model.
# The Agent Loop as a function
def run_agent_loop(initial_memory, initial_iterations, max_iterations_val, agent_rules_val):
    memory = list(initial_memory) # Create a mutable copy
    iterations = initial_iterations
    agent_rules = agent_rules_val
    max_iterations = max_iterations_val

    while iterations < max_iterations:
        # 1. Construct prompt: Combine agent rules with memory
        prompt = agent_rules + memory

        # 2. Generate response from LLM
        print("Agent thinking...")
        response, response_log = generate_response(prompt) #Get a response
        print(f"Agent response: {response}")

        # 3. Parse response to determine action
        action = parse_action(response)

        result = "Action executed"

        if action["tool_name"] == "list_files":
            result = {"result": list_files()}
        elif action["tool_name"] == "read_file":
            result = {"result": read_file(action["args"]["file_name"])}
        elif action["tool_name"] == "error":
            result = {"error": action["args"]["message"]}
        elif action["tool_name"] == "terminate":
            print(action["args"]["message"])
            break
        else:
            result = {"error": "Unknown action: " + action["tool_name"]}

        print(f"Action result: {result}")

        # 5. Update memory with response and results
        memory.extend([
            {"role": "assistant", "content": response},
            {"role": "user", "content": json.dumps(result)}
        ])

        # 6. Check termination condition
        if action["tool_name"] == "terminate":
            break

        iterations += 1

    print(f"Agent loop finished after {iterations} iterations.")
    return memory

## Step 1: Constructing the Agent Prompt

The prompt is created by appending the agent’s rules (system message) to the current memory of interactions. Part of the memory is a descripton of the task that the agent should perofrm. This ensures the agent is always aware of its tools and constraints while also remembering past actions.

```python
prompt = agent_rules + memory
```

**Explanation:**

*   **agent_rules**: This contains the predefined system instructions, ensuring the agent behaves within its defined constraints and understands its tools.
*   **memory**: This is a record of all past interactions, including user input, the agent’s responses, and the results of executed actions.

By constructing the prompt this way, the agent retains continuity across iterations, ensuring it can adapt its behavior based on previous actions and results. The memory tells it what just happened, what happened in the past, and informs its decision of the next action.

---

*Migration Note: This approach remains consistent with both OpenAI and Gemini models, as it focuses on context construction for the LLM.*

### Agent Rules: Defining the Agent’s Behavior

Before the agent begins its loop, it must have a clear set of rules that define its behavior, capabilities, and constraints. These agent rules are specified in the system message and play a critical role in ensuring the agent interacts predictably and within its defined boundaries.

**How it works in code:**

The `agent_rules` are written as a system message that instructs the LLM on how the agent should behave, what tools it has available, and how to format its responses. These rules are included at the start of the prompt for every iteration.

### **Agent Rules: Defining the Agent’s Behavior**

Before the agent begins its loop, it must have a clear set of rules that define its behavior, capabilities, and constraints. These `agent rules` are specified in the system message and play a critical role in ensuring the agent interacts predictably and within its defined boundaries.

In [6]:
# Migrated from OpenAI to Gemini model.
agent_rules = [{
    "role": "system",
    "content": """
You are an AI agent that can perform tasks by using available tools.
When you perform an action, its result will be provided to you in a subsequent 'user' turn, formatted as a JSON string. You should interpret these 'user' turns as tool outputs, not new user requests, unless explicitly stated by the user.

Available tools:
- list_files() -> List[str]: List all files in the current directory.
- read_file(file_name: str) -> str: Read the content of a file.
- terminate(message: str): End the agent loop and print a summary to the user.

If a user asks about files, list them using `list_files()` before attempting to read. After listing files, if the user has not specified a file to read, you should inform the user of the available files and ask them which one they'd like to read. If you have already listed the files and there are no further explicit instructions from the user on what to do next, you should use the `terminate` tool with an appropriate message summarizing the available files.

Every response MUST have an action.
Respond in this format:

```action
{
    "tool_name": "insert tool_name",
    "args": {...fill in any required arguments here...}
}
"""
}]

**Explanation:**

*   **Role of system messages**: The system role in the messages list is used to establish ground rules for the agent. This ensures the LLM understands what it can do and how it should behave throughout the session.
*   **Tools description**: The agent rules explicitly list the tools the agent can use, providing a structured interface for interaction with the environment.
*   **Output format**: The rules enforce a standardized output format ("```action {...}"), which makes parsing and executing actions easier and less error-prone.

Each of the “tools” in the system prompt correspond to a function in the code. The agent is going to choose what function to execute and when. Moreover, it is going to decide the parameters that are provided to the functions.

The agent is not creating the functions at this point; it is orchestrating their behavior. This means that the logic for how each tool operates is predefined in the code, and the agent focuses on selecting the right tool for the job and providing the correct input to that tool.

Because agents can adapt as the loop progresses, they can dynamically decide which tool to use based on the current context and task requirements. This ability allows the agent to adjust its behavior as new information becomes available, making it more flexible and responsive to the user’s input.

For example, if the user asks the agent to read the contents of a specific file, the agent will first use the 'list_files' tool to identify the available files. Then, based on the result, it will determine whether to proceed with the 'read_file' tool or respond with an error if the file does not exist. The agent evaluates each step iteratively, ensuring its actions are informed by the current state of the environment.

This orchestration process, driven by the **agent rules** and the tools available, showcases the power of combining pre-defined functions with adaptive decision-making. By allowing the agent to focus on **what to do** rather than **how to do** it, we create a system that leverages the LLM for high-level reasoning while relying on well-defined code for execution.

This separation of reasoning and execution is what makes the agent loop so powerful—it creates a modular, extensible framework that can handle increasingly complex tasks without rewriting the underlying tools.

Additionally, the agent loop eliminates much of the “glue code” traditionally required to tie these fundamental functions together. Instead of hardcoding workflows, the agent dynamically decides the sequence of actions needed to achieve a task, effectively realizing a program on top of its components. This dynamic nature enables the agent to combine its tools in ways that would typically require custom logic, making it far more versatile and capable of addressing a broader range of use cases without additional development overhead.

### Example in practice:

If the user asks, “What files are here?”, the agent rules guide the LLM to respond with something like:

In [7]:
{
    "tool_name": "list_files",
    "args": {}
}

{'tool_name': 'list_files', 'args': {}}

This response ensures the agent’s next step is both predictable and executable within its predefined constraints.

**How `agent_rules` integrate with the loop:**

The `agent_rules` are combined with the `memory` in `Step 1: Construct Prompt` to form the input for the LLM. This guarantees that the agent always has access to its instructions and tools at every iteration. We will discuss the memory in more detail later.

This step prepares the input for the LLM by combining the system rules and the memory of the agent’s previous interactions. The goal is to give the LLM all the necessary context for generating the next action.

### Example in practice:

If the user asks, “What files are in this directory?”, the memory might look like this:

In [8]:
# Migrated from OpenAI to Gemini model.
memory = [
    {"role": "user", "content": "What files are in this directory?"},
    {"role": "assistant", "content": "```action\n{\"tool_name\":\"list_files\",\"args\":{}}\n```"},
    {"role": "user", "content": "[\"file1.txt\", \"file2.txt\"]"}
]

Adding `agent_rules` ensures the LLM understands what tools it can use to continue interacting.

## Step 2: Generate Response

After constructing the prompt, the agent sends it to the LLM to receive a response. This response will define the next action for the agent to execute.

**Code snippet:**

In [9]:
prompt = agent_rules + memory
response, response_log = generate_response(prompt)
print(f"Agent response: {response}")

Agent response: The directory contains the following files: `file1.txt` and `file2.txt`. Which one would you like to read?

```action
{"tool_name": "terminate", "args": {"message": "The available files are file1.txt and file2.txt. Please specify which file you would like to read."}}
```


**Explanation:**

The `generate_response` function sends the prompt to the Gemini model and retrieves its response. The response typically includes a structured action that the agent will parse and execute in the next steps. This is where the LLM decides what action the agent should take, based on the provided context and rules.

<br>

***
---

<br>

Once the Agent has generated a response, we need to interface the agent with **the environment**. This involves figuring out how the Agent’s response corresponds to an action in **the environment**. Once the correct action is determined, the interface can execute the action and later provide the Agent feedback on the result of the action.

## Step 3: Parse the Response
After generating a response, the next step is to extract the intended action and its parameters from the LLM’s output. The response is expected to follow a predefined structure, such as a JSON format encapsulated within a markdown code block. This structure ensures the action can be parsed and executed without ambiguity.

In the code, this is accomplished by locating and extracting the content between the ```action markers. If the response does not include a valid action block, the agent defaults to a termination action, returning the raw response as the message:



In [10]:
def parse_action(response: str) -> Dict:
    """Parse the LLM response into a structured action dictionary."""
    try:
        response = extract_markdown_block(response, "action")
        response_json = json.loads(response)
        if "tool_name" in response_json and "args" in response_json:
            return response_json
        else:
            return {"tool_name": "error", "args": {"message": "You must respond with a JSON tool invocation."}}
    except json.JSONDecodeError:
        return {"tool_name": "error", "args": {"message": "Invalid JSON response. You must respond with a JSON tool invocation."}}


In [11]:
import re

# This function was not provided with the starter code.
def extract_markdown_block(text, block_name):

    """Extracts content from a markdown code block with the given block_name."""
    # Regex to find a code block with the specific block_name
    pattern = rf'```{re.escape(block_name)}\s*\n(.*?)\n```'
    match = re.search(pattern, text, re.DOTALL)
    if match:
        return match.group(1).strip()
    return None

<br>This parsing step is critical to ensuring the response is actionable. It provides a structured output, such as:

In [12]:
{
    "tool_name": "list_files",
    "args": {}
}

{'tool_name': 'list_files', 'args': {}}

By breaking down the LLM’s output into tool_name and args, the agent can precisely determine the next action and its inputs.

If the LLM response does not contain a valid action block, the agent defaults to an error message, prompting the LLM to provide a valid JSON tool invocation. The error message appears to have come from the “user”. This fallback mechanism ensures the agent can recover if it starts outputting invalid responses that aren’t in the desired format.

## Step 4: Execute the Action
Once the response is parsed, the agent uses the extracted tool_name and args to execute the corresponding function. Each predefined tool in the system instructions corresponds to a specific function in the code, enabling the agent to interact with its environment.

The execution logic involves mapping the tool_name to the appropriate function and passing the provided arguments:

```python
if action["tool_name"] == "list_files":
    result = {"result": list_files()}
elif action["tool_name"] == "read_file":
    result = {"result": read_file(action["args"]["file_name"])}
elif action["tool_name"] == "error":
    result = {"error": action["args"]["message"]}
elif action["tool_name"] == "terminate":
    print(action["args"]["message"])
    break
else:
    result = {"error":"Unknown action: "+action["tool_name"]}
```

For example, if the action specifies tool_name as list_files with empty args, the list_files() function is called, and the agent returns the list of files in the directory. Similarly, a read_file action extracts the filename from the arguments and retrieves its content.

The execution step is the point where the agent performs tangible work, such as interacting with files or printing messages to the console. It bridges the decision-making process with concrete results that feed back into the agent’s memory for subsequent iterations.

## Step 5: Update the Agent’s Memory
After executing an action, the agent updates its memory with the results. Memory serves as the agent’s record of what has happened during the interaction, including user requests, the actions performed, and their outcomes. By appending this information to the memory, the agent retains context, enabling it to make more informed decisions in future iterations.

In the code, memory is updated by extending it with both the LLM’s response (representing the agent’s intention) and the result of the executed action:

```python
memory.extend([
    {"role": "assistant", "content": response},
    {"role": "user", "content": json.dumps(result)}
])
```

###How This Works:
- The assistant role captures the structured response generated by the LLM.
- The user role captures the feedback in the form of the action result, ensuring that the LLM has a clear understanding of what happened after the action was performed. The results of actions are always communicated back to the LLM with the “user” role.

By keeping a running history of these exchanges, the agent maintains continuity, allowing it to refine its behavior dynamically as the memory grows and track the status of its work.

## Step 6: Decide Whether to Continue
The final step in each iteration of the agent loop is determining whether to continue or terminate. This decision is based on the action executed and the state of the task at hand. If the parsed action specifies terminate, or if a predefined condition (e.g., maximum iterations) is met, the agent ends its loop.

In the code, this is implemented as a simple conditional check:

```python
if action["tool_name"] == "terminate":
    print(action["args"]["message"])
    break
```

If the action specifies a termination, the loop exits, and the agent provides a closing message defined in the terminate action’s arguments. If no termination is triggered, the agent loops back to process the next user request or continue its task.

##Example: Iterative Adaptation
Imagine the agent is tasked with reading a file but encounters a missing filename in the initial request.

1. In the first iteration, it executes list_files to retrieve the available files.
2. Based on the memory of this result, it refines its next action, prompting the user to select a specific file.
3. This iterative process continues until the task is completed or the agent determines that no further actions are required.

Each loop iteration, the agent can look back at its memory to decide if it has completed the overall task. The memory is a critical part of deciding if the agent should continue or terminate. By deciding whether to continue at each step, the agent balances its ability to dynamically adapt to new information with the need to eventually conclude its task. The agent can also be instructed on when to terminate the loop, such as if more than two errors are encountered or if a specific condition is met.

In [13]:
# Call the agent loop function
final_memory = run_agent_loop(memory, iterations, max_iterations, agent_rules)
print("Final memory after agent loop:")
for entry in final_memory:
    print(entry)

Agent thinking...
Agent response: I have found the following files in the directory: file1.txt and file2.txt. Which one would you like to read?

```action
{"tool_name": "terminate", "args": {"message": "The available files are file1.txt and file2.txt. Please let me know if you would like to read one of them."}}
```
The available files are file1.txt and file2.txt. Please let me know if you would like to read one of them.
Agent loop finished after 0 iterations.
Final memory after agent loop:
{'role': 'user', 'content': 'What files are in this directory?'}
{'role': 'assistant', 'content': '```action\n{"tool_name":"list_files","args":{}}\n```'}
{'role': 'user', 'content': '["file1.txt", "file2.txt"]'}


## Starter Code Source:

*  [**Specialization**: AI Agent Developer Specialization:](https://www.coursera.org/specializations/ai-agents)
*   [**Course**: AI Agents and Agentic AI with Python & Generative AI | Coursera](https://www.coursera.org/learn/ai-agents-python)